In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

df = pd.read_csv(
    '/kaggle/input/datasets/uciml/sms-spam-collection-dataset/spam.csv',
    encoding='latin-1'
)

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns = ['Unnamed: 2','Unnamed: 3','Unnamed: 4'])

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates()

In [ ]:
df = df.rename(columns={
    'v1': 'target',
    'v2': 'text'
})
print(df.sample(10))

In [ ]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
df['target']=label.fit_transform(df['target'])

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

In [ ]:
df.sample(10)

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

df['tokens'] = df['text'].apply(word_tokenize)

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Preserve negation words
stop_words -= {'not', 'no', 'nor'}

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens'] = df['tokens'].apply(remove_stopwords)

print(df['tokens'].head())

In [ ]:
df.sample(10)

In [ ]:
df['processed_text'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))

In [ ]:
X = df['processed_text']
y = df['target']

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size = 0.2,
    random_state = 42,
    stratify = y)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer 

vectorizer = TfidfVectorizer(
     max_features=10000,
    ngram_range=(1, 2)
)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
max_length = 100

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq, maxlen=max_length, padding="post", truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq, maxlen=max_length, padding="post", truncating="post"
)

print("Training shape:", X_train_pad.shape)
print("Testing shape:", X_test_pad.shape)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

rnn_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True),
    SimpleRNN(64),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_model.summary()

In [ ]:
history_rnn = rnn_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]
)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_prob_rnn = rnn_model.predict(X_test_pad).ravel()
y_pred_rnn = (y_prob_rnn >= 0.5).astype(int)

print("RNN Accuracy:", accuracy_score(y_test, y_pred_rnn))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rnn))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rnn))

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [ ]:
import numpy as np

print("Training shape:", X_train_pad.shape)
print("First 5 sequences:")
print(X_train_pad[:5])

def is_right_padded(row):
    zeros = np.where(row == 0)[0]

    if len(zeros) == 0:
        return True

    return np.all(row[zeros[0]:] == 0)

print(
    "All sequences are right-padded:",
    all(is_right_padded(row) for row in X_train_pad)
)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import tensorflow as tf

lstm_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64, use_cudnn=False),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_lstm = lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]
)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_prob_lstm = lstm_model.predict(X_test_pad).ravel()
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

print("LSTM Accuracy:", accuracy_score(y_test, y_pred_lstm))
print(confusion_matrix(y_test, y_pred_lstm))
print(classification_report(y_test, y_pred_lstm))

In [ ]:
from sklearn.metrics import classification_report

print("NAIVE BAYES")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

print("RNN")
print(classification_report(y_test, y_pred_rnn, target_names=["Ham", "Spam"]))

print("LSTM")
print(classification_report(y_test, y_pred_lstm, target_names=["Ham", "Spam"]))

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

gru_model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    GRU(64, use_cudnn=False),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_gru = gru_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]
)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_prob_gru = gru_model.predict(X_test_pad).ravel()
y_pred_gru = (y_prob_gru >= 0.5).astype(int)

print("GRU Accuracy:", accuracy_score(y_test, y_pred_gru))

print(classification_report(
    y_test,
    y_pred_gru,
    target_names=["Ham", "Spam"]
))